In [2]:
# Import libraries and StackSats classes needed for exporting strategy weights, merging data, and plotting results.
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.baselines.uniform import UniformStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

In [3]:
# Initialize the runner and load the prepared Bitcoin analytics parquet manually.
runner = StrategyRunner()

btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"
btc_df = pl.read_parquet(btc_path).with_columns(pl.col("date").cast(pl.Datetime))

print("Loaded rows:", btc_df.height)
print(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Loaded rows: 5689
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [4]:
btc_full = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_full.height)
print(
    btc_full.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [5]:
# Define the 4 calendar-style cycles for analysis.
# Each cycle ends on Dec 31.
calendar_cycles = [
    {
        "label": "Cycle 1: 2010-2013",
        "start": "2010-08-16",
        "end": "2013-12-31"
    },
    {
        "label": "Cycle 2: 2014-2017",
        "start": "2014-01-01",
        "end": "2017-12-31"
    },
    {
        "label": "Cycle 3: 2018-2021",
        "start": "2018-01-01",
        "end": "2021-12-31"
    },
    {
        "label": "Cycle 4: 2022-2023",
        "start": "2022-01-01",
        "end": "2023-12-31"
    }
]

calendar_cycles

[{'label': 'Cycle 1: 2010-2013', 'start': '2010-08-16', 'end': '2013-12-31'},
 {'label': 'Cycle 2: 2014-2017', 'start': '2014-01-01', 'end': '2017-12-31'},
 {'label': 'Cycle 3: 2018-2021', 'start': '2018-01-01', 'end': '2021-12-31'},
 {'label': 'Cycle 4: 2022-2023', 'start': '2022-01-01', 'end': '2023-12-31'}]

In [6]:
# Actual Bitcoin halving periods used only for background shading.
# These do not control the export/calculation logic.
halving_periods = [
    {
        "label": "Pre-2012 Halving",
        "start": "2010-08-16",
        "end": "2012-11-27",
        "color": "rgba(173, 216, 230, 0.25)"
    },
    {
        "label": "2012-2016 Halving Cycle",
        "start": "2012-11-28",
        "end": "2016-07-08",
        "color": "rgba(144, 238, 144, 0.25)"
    },
    {
        "label": "2016-2020 Halving Cycle",
        "start": "2016-07-09",
        "end": "2020-05-10",
        "color": "rgba(176, 196, 222, 0.25)"
    },
    {
        "label": "2020-2024 Halving Cycle",
        "start": "2020-05-11",
        "end": "2023-12-31",
        "color": "rgba(238, 232, 170, 0.25)"
    }
]

halving_periods

[{'label': 'Pre-2012 Halving',
  'start': '2010-08-16',
  'end': '2012-11-27',
  'color': 'rgba(173, 216, 230, 0.25)'},
 {'label': '2012-2016 Halving Cycle',
  'start': '2012-11-28',
  'end': '2016-07-08',
  'color': 'rgba(144, 238, 144, 0.25)'},
 {'label': '2016-2020 Halving Cycle',
  'start': '2016-07-09',
  'end': '2020-05-10',
  'color': 'rgba(176, 196, 222, 0.25)'},
 {'label': '2020-2024 Halving Cycle',
  'start': '2020-05-11',
  'end': '2023-12-31',
  'color': 'rgba(238, 232, 170, 0.25)'}]

In [7]:
def export_one_year(
    strategy,
    btc_data: pl.DataFrame,
    year: int,
    runner: StrategyRunner
) -> pl.DataFrame | None:
    
    year_start = f"{year}-01-01"
    year_end = f"{year}-12-31"

    year_df = (
        btc_data
        .filter(
            (pl.col("date") >= pl.datetime(year, 1, 1)) &
            (pl.col("date") <= pl.datetime(year, 12, 31))
        )
        .sort("date")
    )

    if year_df.is_empty():
        print(f"{year}: skipped, no data")
        return None

    if year_df.height < 365:
        print(f"{year}: skipped, less than 365 rows")
        return None

    try:
        config = ExportConfig(
            range_start=year_start,
            range_end=year_end
        )

        export_obj = runner.export(strategy, config, btc_df=year_df)
        df = export_obj.to_dataframe()

        df = df.with_columns([
            pl.col("start_date").cast(pl.Datetime),
            pl.col("end_date").cast(pl.Datetime),
            pl.col("date").cast(pl.Datetime),
        ])

        # For leap years, export may return two 365-day windows.
        # Keep the one ending latest.
        latest_end = df.select(pl.col("end_date").max()).item()

        df_one = (
            df
            .filter(pl.col("end_date") == latest_end)
            .sort("date")
        )

        print(f"{year}: exported {df_one.height} rows")
        return df_one

    except Exception as e:
        print(f"{year}: skipped due to error -> {e}")
        return None

In [8]:
import pandas as pd
def process_cycle_year_by_year(
    cycle: dict,
    btc_data: pl.DataFrame,
    runner: StrategyRunner,
    total_budget_usd: float = 1000.0,
    top_buy_quantile: float = 0.90
):
    cycle_label = cycle["label"]
    cycle_start = cycle["start"]
    cycle_end = cycle["end"]

    start_year = int(cycle_start[:4])
    end_year = int(cycle_end[:4])

    print(f"\nProcessing {cycle_label}")

    cycle_df = (
        btc_data
        .filter(
            (pl.col("date") >= pl.lit(cycle_start).str.to_datetime()) &
            (pl.col("date") <= pl.lit(cycle_end).str.to_datetime())
        )
        .sort("date")
    )

    print(
        cycle_df.select(
            pl.col("date").min().alias("min_date"),
            pl.col("date").max().alias("max_date"),
            pl.len().alias("rows")
        )
    )

    momentum_strategy = MomentumStrategy()
    uniform_strategy = UniformStrategy()

    momentum_yearly = []
    uniform_yearly = []

    for year in range(start_year, end_year + 1):
        momentum_result = export_one_year(momentum_strategy, cycle_df, year, runner)
        if momentum_result is not None:
            momentum_yearly.append(momentum_result)

        uniform_result = export_one_year(uniform_strategy, cycle_df, year, runner)
        if uniform_result is not None:
            uniform_yearly.append(uniform_result)

    if not momentum_yearly:
        raise ValueError(f"No valid Momentum exports for {cycle_label}")

    if not uniform_yearly:
        raise ValueError(f"No valid Uniform exports for {cycle_label}")

    momentum_all = pl.concat(momentum_yearly).rename({"weight": "momentum_weight_raw"})
    uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight_raw"})

    merged = (
        momentum_all
        .select(["date", "price_usd", "momentum_weight_raw"])
        .join(
            uniform_all.select(["date", "baseline_weight_raw"]),
            on="date",
            how="inner"
        )
        .sort("date")
    )

    # Normalize both strategies inside this cycle.
    momentum_sum = merged["momentum_weight_raw"].sum()
    baseline_sum = merged["baseline_weight_raw"].sum()

    merged = merged.with_columns([
        (pl.col("momentum_weight_raw") / momentum_sum).alias("momentum_weight"),
        (pl.col("baseline_weight_raw") / baseline_sum).alias("baseline_weight"),
    ])

    # Convert weights into USD allocation.
    merged = merged.with_columns([
        (pl.col("momentum_weight") * total_budget_usd).alias("dynamic_usd"),
        (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
    ])

    # Convert USD allocation into BTC.
    merged = merged.with_columns([
        (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
        (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
    ])

    # Convert BTC into sats.
    merged = merged.with_columns([
        (pl.col("btc_accum_dynamic") * 100_000_000).alias("sats_accum_dynamic"),
        (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
    ])

    # Daily sats per dollar.
    merged = merged.with_columns([
        (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
        (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
    ])

    total_dynamic_btc = merged["btc_accum_dynamic"].sum()
    total_baseline_btc = merged["btc_accum_baseline"].sum()

    sats_per_dollar_dynamic = (total_dynamic_btc / total_budget_usd) * 100_000_000
    sats_per_dollar_baseline = (total_baseline_btc / total_budget_usd) * 100_000_000

    pct_diff_vs_baseline = (
        (total_dynamic_btc - total_baseline_btc) / total_baseline_btc
    ) * 100

    performance_label = "better" if pct_diff_vs_baseline > 0 else "worse"

    # Top buy days = top 10% of Momentum weights within this cycle.
    top_buy_threshold = merged["momentum_weight"].quantile(top_buy_quantile)

    top_buy_points = (
        merged
        .filter(pl.col("momentum_weight") >= top_buy_threshold)
        .select(["date", "price_usd", "momentum_weight"])
        .sort("momentum_weight", descending=True)
        .to_pandas()
    )

    top_buy_points["date"] = pd.to_datetime(top_buy_points["date"])

    plot_df = merged.to_pandas()
    plot_df["date"] = pd.to_datetime(plot_df["date"])

    return {
        "label": cycle_label,
        "start": cycle_start,
        "end": cycle_end,
        "merged": merged,
        "plot_df": plot_df,
        "top_buy_points": top_buy_points,
        "top_buy_threshold": top_buy_threshold,
        "top_buy_quantile": top_buy_quantile,
        "total_dynamic_btc": total_dynamic_btc,
        "total_baseline_btc": total_baseline_btc,
        "sats_per_dollar_dynamic": sats_per_dollar_dynamic,
        "sats_per_dollar_baseline": sats_per_dollar_baseline,
        "pct_diff_vs_baseline": pct_diff_vs_baseline,
        "performance_label": performance_label,
    }

In [9]:
cycle_results = {}

for cycle in calendar_cycles:
    result = process_cycle_year_by_year(
        cycle=cycle,
        btc_data=btc_full,
        runner=runner,
        total_budget_usd=1000.0,
        top_buy_quantile=0.90
    )

    cycle_results[cycle["label"]] = result


Processing Cycle 1: 2010-2013
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2010-08-16 00:00:00 ┆ 2013-12-31 00:00:00 ┆ 1234 │
└─────────────────────┴─────────────────────┴──────┘
2010: skipped, less than 365 rows
2010: skipped, less than 365 rows


2011: exported 365 rows
2011: exported 365 rows
2012: exported 365 rows
2012: exported 365 rows
2013: exported 365 rows
2013: exported 365 rows

Processing Cycle 2: 2014-2017
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2014-01-01 00:00:00 ┆ 2017-12-31 00:00:00 ┆ 1461 │
└─────────────────────┴─────────────────────┴──────┘
2014: exported 365 rows
2014: exported 365 rows
2015: exported 365 rows
2015: exported 365 rows
2016: exported 365 rows
2016: exported 365 rows
2017: exported 365 rows
2017: exported 365 rows

Processing Cycle 3: 2018-2021
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        

In [10]:
# Create one interactive Plotly chart for one cycle.
# This uses the year-by-year export result, so it should match the static chart methodology.
import plotly.graph_objects as go
def plot_interactive_cycle_yearly(result: dict, halving_periods: list[dict]):
    plot_df = result["plot_df"].copy()
    top_buy_points = result["top_buy_points"].copy()

    cycle_start = pd.to_datetime(result["start"])
    cycle_end = pd.to_datetime(result["end"])

    fig = go.Figure()

    # Actual halving-period background shading, clipped to this cycle.
    for period in halving_periods:
        period_start = pd.to_datetime(period["start"])
        period_end = pd.to_datetime(period["end"])

        shaded_start = max(period_start, cycle_start)
        shaded_end = min(period_end, cycle_end)

        if shaded_start <= shaded_end:
            fig.add_vrect(
                x0=shaded_start,
                x1=shaded_end,
                fillcolor=period["color"],
                opacity=1.0,
                layer="below",
                line_width=0,
                annotation_text=period["label"],
                annotation_position="top left",
                annotation_font_size=11,
            )

    # BTC price line.
    fig.add_trace(
        go.Scatter(
            x=plot_df["date"],
            y=plot_df["price_usd"],
            mode="lines",
            name="BTC Price (USD)",
            line=dict(color="black", width=2),
            yaxis="y1",
            customdata=plot_df[[
                "momentum_weight",
                "baseline_weight",
                "sats_per_dollar_dynamic",
                "sats_per_dollar_baseline",
                "sats_accum_dynamic",
                "sats_accum_baseline",
            ]],
            hovertemplate=(
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>BTC Price</b>: $%{y:,.2f}<br>"
                "<b>Momentum Weight</b>: %{customdata[0]:.8f}<br>"
                "<b>DCA Weight</b>: %{customdata[1]:.8f}<br>"
                "<b>Momentum sats/$</b>: %{customdata[2]:,.2f}<br>"
                "<b>DCA sats/$</b>: %{customdata[3]:,.2f}<br>"
                "<b>Momentum sats accumulated</b>: %{customdata[4]:,.2f}<br>"
                "<b>DCA sats accumulated</b>: %{customdata[5]:,.2f}"
                "<extra></extra>"
            )
        )
    )

    # Momentum allocation area.
    fig.add_trace(
        go.Scatter(
            x=plot_df["date"],
            y=plot_df["momentum_weight"],
            mode="lines",
            name="Momentum Weight",
            line=dict(color="green", width=1.5),
            fill="tozeroy",
            fillcolor="rgba(0, 128, 0, 0.35)",
            yaxis="y2",
            customdata=plot_df[[
                "price_usd",
                "sats_per_dollar_dynamic",
                "sats_accum_dynamic",
            ]],
            hovertemplate=(
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>Momentum Weight</b>: %{y:.8f}<br>"
                "<b>BTC Price</b>: $%{customdata[0]:,.2f}<br>"
                "<b>Momentum sats/$</b>: %{customdata[1]:,.2f}<br>"
                "<b>Momentum sats accumulated</b>: %{customdata[2]:,.2f}"
                "<extra></extra>"
            )
        )
    )

    # Baseline DCA line.
    fig.add_trace(
        go.Scatter(
            x=plot_df["date"],
            y=plot_df["baseline_weight"],
            mode="lines",
            name="Baseline DCA Weight",
            line=dict(color="orange", width=2, dash="dash"),
            yaxis="y2",
            customdata=plot_df[[
                "price_usd",
                "sats_per_dollar_baseline",
                "sats_accum_baseline",
            ]],
            hovertemplate=(
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>DCA Weight</b>: %{y:.8f}<br>"
                "<b>BTC Price</b>: $%{customdata[0]:,.2f}<br>"
                "<b>DCA sats/$</b>: %{customdata[1]:,.2f}<br>"
                "<b>DCA sats accumulated</b>: %{customdata[2]:,.2f}"
                "<extra></extra>"
            )
        )
    )

    # Top 10% buy triangles.
    fig.add_trace(
        go.Scatter(
            x=top_buy_points["date"],
            y=top_buy_points["price_usd"],
            mode="markers",
            name="Top 10% Buy Days",
            marker=dict(
                color="green",
                symbol="triangle-up",
                size=9,
                line=dict(color="black", width=1)
            ),
            yaxis="y1",
            customdata=top_buy_points[["momentum_weight"]],
            hovertemplate=(
                "<b>Top 10% Buy Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>BTC Price</b>: $%{y:,.2f}<br>"
                "<b>Momentum Weight</b>: %{customdata[0]:.8f}"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=(
            f"{result['label']} | Momentum Strategy vs Baseline DCA<br>"
            f"Dynamic BTC: {result['total_dynamic_btc']:.6f} | "
            f"DCA BTC: {result['total_baseline_btc']:.6f} | "
            f"Momentum performed {abs(result['pct_diff_vs_baseline']):.2f}% "
            f"{result['performance_label']} than DCA | "
            f"Top buy threshold: {result['top_buy_threshold']:.8f}"
        ),
        width=1250,
        height=650,
        hovermode="x unified",
        xaxis=dict(
            title="Date",
            range=[cycle_start, cycle_end]
        ),
        yaxis=dict(
            title="BTC Price (USD, log scale)",
            type="log",
            side="left"
        ),
        yaxis2=dict(
            title="Allocation Weight",
            overlaying="y",
            side="right",
            showgrid=False
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        ),
        margin=dict(l=70, r=70, t=120, b=60)
    )

    fig.show()

In [11]:
# Create 4 separate interactive charts, one for each cycle.

for label, result in cycle_results.items():
    plot_interactive_cycle_yearly(result, halving_periods)

In [12]:
# Build monthly table for each cycle.
def build_monthly_table(result: dict):
    merged = result["merged"]

    monthly = (
        merged
        .with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
        .group_by("month")
        .agg([
            pl.col("price_usd").mean().alias("avg_price_usd"),
            pl.col("momentum_weight").sum().alias("total_dynamic_weight"),
            pl.col("baseline_weight").sum().alias("total_baseline_weight"),
            pl.col("sats_accum_dynamic").sum().alias("sats_accum_dynamic"),
            pl.col("sats_accum_baseline").sum().alias("sats_accum_baseline"),
            pl.col("dynamic_usd").sum().alias("total_dynamic_usd"),
            pl.col("baseline_usd").sum().alias("total_baseline_usd"),
        ])
        .with_columns([
            (pl.col("sats_accum_dynamic") / pl.col("total_dynamic_usd")).alias("avg_sats_per_dollar_dynamic"),
            (pl.col("sats_accum_baseline") / pl.col("total_baseline_usd")).alias("avg_sats_per_dollar_baseline"),
        ])
        .sort("month")
    )

    monthly_pd = monthly.to_pandas()
    monthly_pd["month"] = monthly_pd["month"].dt.strftime("%Y-%m")
    monthly_pd["avg_price_usd"] = monthly_pd["avg_price_usd"].round(2)

    for col in [
        "avg_sats_per_dollar_dynamic",
        "avg_sats_per_dollar_baseline",
        "total_dynamic_weight",
        "total_baseline_weight",
        "sats_accum_dynamic",
        "sats_accum_baseline",
    ]:
        monthly_pd[col] = monthly_pd[col].round(4)

    monthly_pd = monthly_pd[[
        "month",
        "avg_price_usd",
        "avg_sats_per_dollar_dynamic",
        "avg_sats_per_dollar_baseline",
        "total_dynamic_weight",
        "total_baseline_weight",
        "sats_accum_dynamic",
        "sats_accum_baseline",
    ]]

    return monthly_pd

In [13]:
# Create monthly tables for all 4 cycles.
cycle_monthly_tables = {
    label: build_monthly_table(result)
    for label, result in cycle_results.items()
}

In [14]:
# Create one combined interactive Plotly chart across all 4 cycles.
# This uses the same year-by-year export results stored in cycle_results.

import plotly.graph_objects as go
import pandas as pd
import polars as pl

# Combine all merged cycle dataframes into one dataframe
combined_merged = pl.concat(
    [result["merged"] for result in cycle_results.values()]
).sort("date")

# Convert to pandas for plotting
combined_plot_df = combined_merged.to_pandas()
combined_plot_df["date"] = pd.to_datetime(combined_plot_df["date"])
combined_plot_df = combined_plot_df.sort_values("date")

# Add cycle label for tooltip
def assign_cycle_label(date):
    for cycle in calendar_cycles:
        start = pd.to_datetime(cycle["start"])
        end = pd.to_datetime(cycle["end"])
        if start <= date <= end:
            return cycle["label"]
    return "Outside cycle"

combined_plot_df["cycle_label"] = combined_plot_df["date"].apply(assign_cycle_label)

# Top buy dates across all cycles combined
top_buy_points_combined = (
    combined_merged
    .sort("momentum_weight", descending=True)
    .head(25)
    .select(["date", "price_usd", "momentum_weight"])
    .to_pandas()
)

top_buy_points_combined["date"] = pd.to_datetime(top_buy_points_combined["date"])

# Overall totals across all cycles
total_dynamic_btc_combined = combined_merged["btc_accum_dynamic"].sum()
total_baseline_btc_combined = combined_merged["btc_accum_baseline"].sum()

# Each cycle used 1000 USD in your earlier logic
total_budget_usd_combined = 1000.0 * len(cycle_results)

sats_per_dollar_dynamic_combined = (
    total_dynamic_btc_combined / total_budget_usd_combined
) * 100_000_000

sats_per_dollar_baseline_combined = (
    total_baseline_btc_combined / total_budget_usd_combined
) * 100_000_000

pct_diff_vs_baseline_combined = (
    (total_dynamic_btc_combined - total_baseline_btc_combined)
    / total_baseline_btc_combined
) * 100

performance_label_combined = (
    "better" if pct_diff_vs_baseline_combined > 0 else "worse"
)

# Create figure
fig = go.Figure()

# Add actual halving-period background shading
for period in halving_periods:
    fig.add_vrect(
        x0=pd.to_datetime(period["start"]),
        x1=pd.to_datetime(period["end"]),
        fillcolor=period["color"],
        opacity=1.0,
        layer="below",
        line_width=0,
        annotation_text=period["label"],
        annotation_position="top left",
        annotation_font_size=11,
    )

# Add vertical lines for cycle boundaries
for cycle in calendar_cycles:
    fig.add_vline(
        x=pd.to_datetime(cycle["start"]),
        line_width=1,
        line_dash="dot",
        line_color="gray"
    )

# BTC price trace on left y-axis (log scale)
fig.add_trace(
    go.Scatter(
        x=combined_plot_df["date"],
        y=combined_plot_df["price_usd"],
        mode="lines",
        name="BTC Price (USD)",
        line=dict(color="black", width=2),
        yaxis="y1",
        customdata=combined_plot_df[
            [
                "cycle_label",
                "momentum_weight",
                "baseline_weight",
                "sats_per_dollar_dynamic",
                "sats_per_dollar_baseline",
                "sats_accum_dynamic",
                "sats_accum_baseline",
            ]
        ],
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Cycle</b>: %{customdata[0]}<br>"
            "<b>BTC Price</b>: $%{y:,.2f}<br>"
            "<b>Momentum Weight</b>: %{customdata[1]:.8f}<br>"
            "<b>DCA Weight</b>: %{customdata[2]:.8f}<br>"
            "<b>Momentum sats/$</b>: %{customdata[3]:,.2f}<br>"
            "<b>DCA sats/$</b>: %{customdata[4]:,.2f}<br>"
            "<b>Momentum sats accumulated</b>: %{customdata[5]:,.2f}<br>"
            "<b>DCA sats accumulated</b>: %{customdata[6]:,.2f}"
            "<extra></extra>"
        ),
    )
)

# Momentum weight on right y-axis
fig.add_trace(
    go.Scatter(
        x=combined_plot_df["date"],
        y=combined_plot_df["momentum_weight"],
        mode="lines",
        name="Momentum Weight",
        line=dict(color="green", width=1.5),
        fill="tozeroy",
        fillcolor="rgba(0, 128, 0, 0.35)",
        yaxis="y2",
        customdata=combined_plot_df[
            [
                "cycle_label",
                "price_usd",
                "sats_per_dollar_dynamic",
                "sats_accum_dynamic",
            ]
        ],
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Cycle</b>: %{customdata[0]}<br>"
            "<b>Momentum Weight</b>: %{y:.8f}<br>"
            "<b>BTC Price</b>: $%{customdata[1]:,.2f}<br>"
            "<b>Momentum sats/$</b>: %{customdata[2]:,.2f}<br>"
            "<b>Momentum sats accumulated</b>: %{customdata[3]:,.2f}"
            "<extra></extra>"
        ),
    )
)

# Baseline DCA on right y-axis
fig.add_trace(
    go.Scatter(
        x=combined_plot_df["date"],
        y=combined_plot_df["baseline_weight"],
        mode="lines",
        name="Baseline DCA Weight",
        line=dict(color="orange", width=2, dash="dash"),
        yaxis="y2",
        customdata=combined_plot_df[
            [
                "cycle_label",
                "price_usd",
                "sats_per_dollar_baseline",
                "sats_accum_baseline",
            ]
        ],
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Cycle</b>: %{customdata[0]}<br>"
            "<b>DCA Weight</b>: %{y:.8f}<br>"
            "<b>BTC Price</b>: $%{customdata[1]:,.2f}<br>"
            "<b>DCA sats/$</b>: %{customdata[2]:,.2f}<br>"
            "<b>DCA sats accumulated</b>: %{customdata[3]:,.2f}"
            "<extra></extra>"
        ),
    )
)

# Top buy dates
fig.add_trace(
    go.Scatter(
        x=top_buy_points_combined["date"],
        y=top_buy_points_combined["price_usd"],
        mode="markers",
        name="Top Buy Dates",
        marker=dict(
            color="green",
            symbol="triangle-up",
            size=11,
            line=dict(color="black", width=1),
        ),
        yaxis="y1",
        customdata=top_buy_points_combined[["momentum_weight"]],
        hovertemplate=(
            "<b>Top Buy Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>BTC Price</b>: $%{y:,.2f}<br>"
            "<b>Momentum Weight</b>: %{customdata[0]:.8f}"
            "<extra></extra>"
        ),
    )
)

# Final layout
fig.update_layout(
    title=(
        "Momentum Strategy vs Baseline DCA Across All 4 Cycles<br>"
        f"Dynamic BTC: {total_dynamic_btc_combined:.6f} | "
        f"DCA BTC: {total_baseline_btc_combined:.6f} | "
        f"Momentum performed {abs(pct_diff_vs_baseline_combined):.2f}% "
        f"{performance_label_combined} than DCA"
    ),
    width=1400,
    height=750,
    hovermode="x unified",
    xaxis=dict(
        title="Date",
        range=[
            pd.to_datetime("2010-08-16"),
            pd.to_datetime("2023-12-31")
        ],
    ),
    yaxis=dict(
        title="BTC Price (USD, log scale)",
        type="log",
        side="left",
    ),
    yaxis2=dict(
        title="Allocation Weight",
        overlaying="y",
        side="right",
        showgrid=False,
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
    ),
    margin=dict(l=70, r=70, t=120, b=60),
)

fig.show()

In [15]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .sort("momentum_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "momentum_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,momentum_weight,baseline_weight
0,2012-12-31,13.24,0.020133,0.000913
1,2013-05-09,111.97,0.002412,0.000913
2,2011-07-09,14.39,0.002234,0.000913
3,2013-05-08,113.47,0.002166,0.000913
4,2011-07-08,14.35,0.002163,0.000913
5,2013-05-03,94.26,0.002053,0.000913
6,2013-05-07,111.43,0.002044,0.000913
7,2011-07-10,15.08,0.001960,0.000913
8,2011-08-06,7.82,0.001959,0.000913
9,2011-08-07,7.72,0.001906,0.000913


In [16]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .filter(pl.col("date").dt.year() == 2011)
    .sort("momentum_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "momentum_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,momentum_weight,baseline_weight
0,2011-07-09,14.39,0.002234,0.000913
1,2011-07-08,14.35,0.002163,0.000913
2,2011-07-10,15.08,0.001960,0.000913
3,2011-08-06,7.82,0.001959,0.000913
4,2011-08-07,7.72,0.001906,0.000913
5,2011-08-08,7.74,0.001897,0.000913
6,2011-09-17,4.77,0.001873,0.000913
7,2011-09-16,4.81,0.001872,0.000913
8,2011-09-15,4.97,0.001866,0.000913
9,2011-09-18,5.11,0.001846,0.000913


In [17]:
(
    cycle1
    .filter(pl.col("date") == pl.datetime(2011, 12, 31))
    .select(["date", "price_usd", "momentum_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,momentum_weight,baseline_weight
0,2011-12-31,4.58,0.000003,0.000913


In [18]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .filter(pl.col("date").dt.year() == 2012)
    .sort("momentum_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "momentum_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,momentum_weight,baseline_weight
0,2012-12-31,13.24,0.020133,0.000913
1,2012-02-19,4.38,0.001239,0.000913
2,2012-02-15,4.69,0.001234,0.000913
3,2012-02-17,4.67,0.001222,0.000913
4,2012-02-14,4.89,0.001215,0.000913
5,2012-02-20,4.44,0.001200,0.000913
6,2012-02-18,4.23,0.001188,0.000913
7,2012-10-26,9.88,0.001179,0.000913
8,2012-02-21,4.58,0.001171,0.000913
9,2012-11-02,10.52,0.001166,0.000913


In [19]:
(
    cycle1
    .filter(pl.col("date") == pl.datetime(2012, 12, 31))
    .select(["date", "price_usd", "momentum_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,momentum_weight,baseline_weight
0,2012-12-31,13.24,0.020133,0.000913


In [20]:
cycle1_2012 = (
    cycle1
    .filter(pl.col("date").dt.year() == 2012)
    .sort("date")
    .with_columns([
        pl.col("momentum_weight").cum_sum().alias("cum_momentum_weight"),
        pl.col("baseline_weight").cum_sum().alias("cum_baseline_weight")
    ])
)

cycle1_2012.tail(10).to_pandas()

,date,price_usd,momentum_weight_raw,baseline_weight_raw,momentum_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cum_momentum_weight,cum_baseline_weight
0,2012-12-22,13.20,0.002609,0.00274,0.000870,0.000913,0.869533,0.913242,0.065874,0.069185,6.587370e+06,6.918500e+06,7.575758e+06,7.575758e+06,0.305998,0.325114
1,2012-12-23,13.14,0.002682,0.00274,0.000894,0.000913,0.894139,0.913242,0.068047,0.069501,6.804711e+06,6.950091e+06,7.610350e+06,7.610350e+06,0.306892,0.326027
2,2012-12-24,13.23,0.002691,0.00274,0.000897,0.000913,0.897107,0.913242,0.067809,0.069028,6.780856e+06,6.902812e+06,7.558579e+06,7.558579e+06,0.307789,0.326941
3,2012-12-25,13.24,0.002703,0.00274,0.000901,0.000913,0.901143,0.913242,0.068062,0.068976,6.806217e+06,6.897598e+06,7.552870e+06,7.552870e+06,0.308690,0.327854
4,2012-12-26,13.18,0.002691,0.00274,0.000897,0.000913,0.896891,0.913242,0.068049,0.069290,6.804940e+06,6.928999e+06,7.587253e+06,7.587253e+06,0.309587,0.328767
5,2012-12-27,13.20,0.002655,0.00274,0.000885,0.000913,0.885124,0.913242,0.067055,0.069185,6.705482e+06,6.918500e+06,7.575758e+06,7.575758e+06,0.310472,0.329680
6,2012-12-28,13.18,0.002696,0.00274,0.000899,0.000913,0.898595,0.913242,0.068179,0.069290,6.817869e+06,6.928999e+06,7.587253e+06,7.587253e+06,0.311371,0.330594
7,2012-12-29,13.11,0.002747,0.00274,0.000916,0.000913,0.915585,0.913242,0.069839,0.069660,6.983866e+06,6.965995e+06,7.627765e+06,7.627765e+06,0.312286,0.331507
8,2012-12-30,13.20,0.002741,0.00274,0.000914,0.000913,0.913640,0.913242,0.069215,0.069185,6.921518e+06,6.918500e+06,7.575758e+06,7.575758e+06,0.313200,0.332420
9,2012-12-31,13.24,0.060400,0.00274,0.020133,0.000913,20.133280,0.913242,1.520640,0.068976,1.520640e+08,6.897598e+06,7.552870e+06,7.552870e+06,0.333333,0.333333


In [21]:
(
    cycle1_2012
    .filter(pl.col("date") < pl.datetime(2012, 12, 31))
    .select([
        pl.col("momentum_weight").sum().alias("momentum_sum_before_last_day"),
        pl.col("baseline_weight").sum().alias("baseline_sum_before_last_day")
    ])
    .to_pandas()
)

,momentum_sum_before_last_day,baseline_sum_before_last_day
0,0.3132,0.33242


In [22]:
cycle1_2011 = (
    cycle1
    .filter(pl.col("date").dt.year() == 2011)
    .sort("date")
    .with_columns([
        pl.col("momentum_weight").cum_sum().alias("cum_momentum_weight"),
        pl.col("baseline_weight").cum_sum().alias("cum_baseline_weight")
    ])
)

cycle1_2011.tail(10).to_pandas()

,date,price_usd,momentum_weight_raw,baseline_weight_raw,momentum_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cum_momentum_weight,cum_baseline_weight
0,2011-12-22,3.79,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000880,0.240961,87950.747587,2.409610e+07,2.638522e+07,2.638522e+07,0.333303,0.325114
1,2011-12-23,3.90,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000855,0.234165,85470.085473,2.341646e+07,2.564103e+07,2.564103e+07,0.333307,0.326027
2,2011-12-24,3.92,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000850,0.232970,85034.013608,2.329699e+07,2.551020e+07,2.551020e+07,0.333310,0.326941
3,2011-12-25,4.14,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000805,0.220590,80515.297928,2.205899e+07,2.415459e+07,2.415459e+07,0.333313,0.327854
4,2011-12-26,4.04,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000825,0.226050,82508.250850,2.260500e+07,2.475248e+07,2.475248e+07,0.333317,0.328767
5,2011-12-27,4.02,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000829,0.227175,82918.739661,2.271746e+07,2.487562e+07,2.487562e+07,0.333320,0.329680
6,2011-12-28,4.14,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000805,0.220590,80515.297928,2.205899e+07,2.415459e+07,2.415459e+07,0.333323,0.330594
7,2011-12-29,4.22,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000790,0.216408,78988.941566,2.164081e+07,2.369668e+07,2.369668e+07,0.333327,0.331507
8,2011-12-30,4.19,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000796,0.217958,79554.494848,2.179575e+07,2.386635e+07,2.386635e+07,0.333330,0.332420
9,2011-12-31,4.58,0.00001,0.00274,0.000003,0.000913,0.003333,0.913242,0.000728,0.199398,72780.203793,1.993978e+07,2.183406e+07,2.183406e+07,0.333333,0.333333


In [23]:
(
    cycle1_2011
    .filter(pl.col("date") < pl.datetime(2012, 12, 31))
    .select([
        pl.col("momentum_weight").sum().alias("momentum_sum_before_last_day"),
        pl.col("baseline_weight").sum().alias("baseline_sum_before_last_day")
    ])
    .to_pandas()
)

,momentum_sum_before_last_day,baseline_sum_before_last_day
0,0.333333,0.333333
